In [0]:
import time

def gen():
  for i in range(10):
    time.sleep(60)
    yield i

In [0]:
import queue, threading, time

TIMEOUT_SECONDS = 2

# def get_timeout():
#     return min(END_TIME - time.time(), TIMEOUT_DELTA)

def producer(g, chunks_queue, end_event):   # producer
    while not end_event.is_set():
        try:
            chunks_queue.put(next(g))
        except StopIteration:
            print("Done!")
            end_event.set()
            chunks_queue.put(None)
            break

def consumer(chunks_queue, end_event):  # consumer
    while not end_event.is_set():
        try:
            chunk = chunks_queue.get(timeout=TIMEOUT_SECONDS)
            if chunk is not None:
              print(chunk)
        except queue.Empty:
            if not end_event.is_set():
              print('Timeout!')
              end_event.set()
            break

g = gen()
chunks_queue = queue.Queue()  # you might wanna use the maxsize parameter
end_event = threading.Event()
producer_thread = threading.Thread(target=producer, args=(g, chunks_queue, end_event))
producer_thread.start()
consumer(chunks_queue, end_event)
producer_thread.join()